# Day 3 · v2 — sequence motifs inside the visit-pruned NMF components (dx_9487)

**Today's goal:** mine ordered motifs *within each of the 8 NMF components* from `Day1_nmf_v2`,
on event streams with every visit event removed (care-driven bias). Same PrefixSpan as v1; what
changes is the input (no visit tokens) and the groups (v2 components instead of KMeans clusters).

Why: Day 1 v2 told us *which codes co-occur*; the open question from the review is *in what order*
for the two competing explanations of dx_9487 (cardiac workup vs anxiety/panic), and whether any
bundle is really about the condition rather than the care setting.


In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 150); pd.set_option('display.max_columns', 40)
import ethos_omop as eo

OUT_DIR = "../../inputs/chest_pain_cohort/"   # matches Day 1's extract_cohort() output dir
DOMAINS = ['Visit', 'Condition', 'Drug', 'Procedure', 'Measurement']   # Visit stays in the load so we can show before vs after

if not os.path.isdir(OUT_DIR):
    raise FileNotFoundError(
        f"{OUT_DIR}/ not found. Run Day 1's eo.extract_cohort(...) (or "
        f"eo.extract_omop(...) with OUT_DIR='omop_extract') first, or "
        f"python make_demo_extract.py for a synthetic stand-in.")

concept  = eo.load_concept(OUT_DIR)
codesets = eo.build_codesets(concept, verbose=False)
raw      = eo.load_events(OUT_DIR, domains=DOMAINS)
person   = eo.load_person(OUT_DIR)
truth    = eo.load_truth(OUT_DIR)                   # None against real data

# v2: drop every visit event BEFORE tagging (same rule as Day1_nmf_v2): a row is a visit if its
# domain OR vocabulary is "visit".
is_visit = (raw['domain'].astype(str).str.lower().eq('visit')
            | raw['vocabulary_id'].astype(str).str.lower().eq('visit'))
visit_ids = set(raw.loc[is_visit, 'concept_id'])
events   = eo.tag_events(raw[~is_visit].reset_index(drop=True), codesets)

tbl = (pd.DataFrame({'before': raw['domain'].value_counts(), 'after': events['domain'].value_counts()})
         .fillna(0).astype(int))
tbl.loc['TOTAL'] = [len(raw), len(events)]
tbl['removed_%'] = ((1 - tbl['after'] / tbl['before']) * 100).round(1)
print("events per domain, BEFORE vs AFTER visit pruning")
print(tbl.to_string())
print(f"\npatients: {raw.person_id.nunique():,} -> {events.person_id.nunique():,}")
print("curated codeset tags on visit concepts (now absent):",
      sorted(t for t, ids in codesets.items() if set(ids) & visit_ids) or "none")
del raw                                              # free memory

print(f"\nevents {events.shape}   patients {events.person_id.nunique()}")
print(f"codeset coverage: {events.in_codeset.mean():.1%} of events carry a clinical tag")
print(f"truth labels: {'present (DEMO extract)' if truth is not None else 'ABSENT (real data)'}")
events.head(6)


In [ ]:
seqs = eo.sequences(events, col='tag', dedup_same_day=True, collapse_repeats=True)
pids = sorted(seqs)
db   = [seqs[p] for p in pids]

print("example timeline:")
print(' -> '.join(db[1][:12]))
print("\nmedian sequence length:", int(np.median([len(s) for s in db])))

### The preprocessing decisions, restated

The first two live in `eo.sequences()` and both are arbitrary:

1. **Same-day ties.** Claims carry a date, not a timestamp, so `DOMAIN_ORDER` breaks ties.
   Visit, condition, procedure, measurement, drug, observation.
2. **Refill collapsing.** Consecutive repeats of the same tag become one token, otherwise
   every pattern is `metformin -> metformin -> metformin`. Chronicity is real signal, so
   capture it as a separate feature — fill count, days supply — not as sequence position.
3. **Visits are not in the sequence at all (v2).** They were the most common tokens in v1 and
   made every motif list a chain of pharmacy / office / ED visits. Dropping them removes that
   care-setting bias; it also removes the `ed_visit` tag, so any motif that depended on "went to the
   ED" is invisible here by construction.

## 1 · PrefixSpan, implemented


In [ ]:
import collections

def prefixspan(db, min_sup, max_len=4):
    """Minimal PrefixSpan over single-item sequences.

    db      : list of lists of tokens
    min_sup : minimum number of sequences containing the pattern
    returns : DataFrame of (pattern, support, support_pct)
    """
    n, results = len(db), []

    def scan(projected, prefix):
        counts = collections.Counter()
        for seq in projected:
            for item in set(seq):                   # count each sequence once
                counts[item] += 1
        for item, c in counts.items():
            if c < min_sup:
                continue
            new_prefix = prefix + [item]
            results.append((tuple(new_prefix), c))
            if len(new_prefix) >= max_len:
                continue
            # project: keep the suffix after the FIRST occurrence of item
            nxt = [seq[seq.index(item) + 1:] for seq in projected if item in seq]
            if nxt:
                scan(nxt, new_prefix)

    scan([list(s) for s in db], [])
    return (pd.DataFrame(results, columns=['pattern', 'support'])
              .assign(support_pct=lambda d: (d.support / n * 100).round(1)))

## 2 · Discriminative mining — motifs per NMF component

Raw support finds whatever is common, which is mostly ambient care. What you want is patterns
common **inside** a group and rare **outside** it. Here the groups are the **8 NMF components of
`Day1_nmf_v2`** (k=8, visits pruned; stable on 4+ of 5 splits).

What the v2 loadings showed, so the motifs below can be read against it:

- **c3 acute / ED workup** owns `troponin` (62%, rank 1); **c0 outpatient cardiometabolic workup** owns
  `ekg`, `stress_test`, `hba1c`, `t2dm`. `cad` and `coronary_angio` are split between the two.
- **No anxiety / panic bundle exists**: `anxiety` is spread 10-22% over c0, c2, c3, c7;
  `panic_attack` sits in c3 (28%) and c7 (27%). `ssri` / `benzodiazepine` load on the prescription
  bundle c1, but so do `beta_blocker` and `statin`.
- **c4** is the GI / screening bundle that owns `gerd`.

Day 3 asks what order those codes come in, per component.


In [ ]:
from scipy import sparse as sp
from sklearn.decomposition import NMF

# Rebuild Day1_nmf_v2's k=8 NMF exactly (same prevalence floor, same anchor drop, same NMF_KW; nndsvd
# init is deterministic), so component numbers match its loadings tables. Day1_nmf_v2 writes the
# cache file below; if it is missing or stale (different extract / floor) this cell fits and saves it.
n_pat      = events.person_id.nunique()
N_TRAIN    = min(20_000, int(0.6 * n_pat))
MIN_PAT    = int(np.ceil(5 * n_pat / N_TRAIN))
prev_pat   = events.groupby('tag').person_id.nunique()
KEEP_TAGS  = prev_pat.index[prev_pat >= MIN_PAT]
nmf_cols   = KEEP_TAGS.drop('chest_pain')                # cohort anchor, 100% prevalence
CACHE      = "nmf_k8_chestpain_v2.npz"                   # same file Day1_nmf_v2 writes

z = np.load(CACHE) if os.path.exists(CACHE) else None
if z is not None and (set(z['cols']) != set(nmf_cols)
                      or set(z['pids']) != set(events.person_id.unique())):
    z = None

if z is not None:
    H   = pd.DataFrame(z['H'], columns=z['cols'])
    lab = pd.Series(z['labels'], index=z['pids'])
    print(f"NMF k=8 loaded from {CACHE}")
else:
    Xb   = eo.bag_of_codes(events[events.tag.isin(KEEP_TAGS)], min_count=1)
    Xs   = sp.csr_matrix(Xb[nmf_cols].to_numpy()).astype(float)
    nmf  = NMF(n_components=8, init='nndsvd', random_state=0, max_iter=2000, tol=1e-4).fit(Xs)
    H    = pd.DataFrame(nmf.components_, columns=nmf_cols)
    lab  = pd.Series(nmf.transform(Xs).argmax(1), index=Xb.index)
    np.savez(CACHE, H=H.to_numpy(), cols=np.array(H.columns, dtype=str), pids=lab.index.to_numpy(),
             labels=lab.to_numpy())
    print(f"NMF k=8 fitted, cached to {CACHE}")

comp = lab.reindex(pids).fillna(-1).astype(int).to_numpy()   # aligned with db / pids

# names read off the Day1_nmf_v2 top-codes and loadings tables; the assert fails loudly if numbering drifted
CARD, RX, NONC, BG = 'cardiac workup', 'drug domain', 'non-cardiac explanation', 'background'
COMP = {0: ('lipid_panel',          'outpatient cardiometabolic / preventive workup (EKG, stress test, HbA1c)', CARD),
        1: ('21_methylprednisolon', 'prescriptions (antibiotics, steroids, NSAIDs, SSRI; beta blockers, statins)', RX),
        2: ('gynecologic_examinat', "women's health (gyn exam, pregnancy test, cytology)",                       BG),
        3: ('troponin',             'acute / ED workup (troponin, radiology, injectables, angiography)',         CARD),
        4: ('level_iv_surgical_pa', 'GI + cancer screening (GERD, endoscopy, colonoscopy, mammography)',        NONC),
        5: ('therapeutic_procedur', 'musculoskeletal / physical therapy',                                       BG),
        6: ('vitamin_b12',          'nutrition + endocrine labs (B12, D, iron, T4)',                            BG),
        7: ('cough',                'acute respiratory infection',                                              BG)}
for c, (prefix, *_) in COMP.items():
    top = H.iloc[c].idxmax()
    assert top.startswith(prefix), (f"component {c} top code is {top!r}, expected {prefix!r}* -- "
                                    f"numbering differs from Day1_nmf_v2; re-read its top codes and fix COMP")

print(f"{'comp':>4} {'n':>6} {'%':>5}  {'role':<24} name")
for c, (_, name, role) in COMP.items():
    n = int((comp == c).sum())
    print(f"{c:>4} {n:>6} {n / len(comp) * 100:5.1f}  {role:<24} {name}")


In [ ]:
from joblib import Parallel, delayed

def contains(seq, pat):
    it = iter(seq)
    return all(x in it for x in pat)

def discriminative(mask, candidates, sequences_db=None, min_in=0.12):
    sdb = sequences_db if sequences_db is not None else db
    out = []
    for p in candidates:
        h = np.array([contains(s, p) for s in sdb])
        si, so = h[mask].mean(), h[~mask].mean()
        if si >= min_in:
            out.append({'pattern': ' -> '.join(p), 'sup_in': round(si, 3),
                        'sup_out': round(so, 3), 'lift': round((si + .01) / (so + .01), 1)})
    if not out:
        return pd.DataFrame(columns=['pattern', 'sup_in', 'sup_out', 'lift'])
    return pd.DataFrame(out).sort_values('lift', ascending=False).reset_index(drop=True)

def _component_job(sdb_in, sdb_out, rel_sup, max_cand):
    """One component: LOCAL mining on its own patients, then lift vs the out-sample.
    Takes only its own slices, so it ships cheaply to a worker process."""
    pats_c = prefixspan(sdb_in, max(3, int(rel_sup * len(sdb_in))), max_len=4)
    pats_c = pats_c[pats_c.pattern.map(len) >= 3].sort_values('support', ascending=False).head(max_cand)
    sdb    = sdb_in + sdb_out
    mask   = np.arange(len(sdb)) < len(sdb_in)
    return discriminative(mask, [tuple(p) for p in pats_c.pattern], sequences_db=sdb, min_in=rel_sup)

def component_motifs(rel_sup=0.10, max_cand=60, n_out=1500, max_in=3000, seed=0, n_jobs=None):
    """All components in parallel (they are independent); largest first so no big job starts last.
    Each component is mined on a random sample of at most `max_in` of its patients (PrefixSpan is
    pure Python; a 30,000-patient component would take hours). Raise max_in once you know the runtime."""
    n_jobs = n_jobs or min(len(COMP), max(1, (os.cpu_count() or 2) - 2))   # leave 2 cores free
    order  = sorted(COMP, key=lambda c: -int((comp == c).sum()))
    jobs   = []
    for c in order:
        idx_in  = np.flatnonzero(comp == c)
        idx_in  = np.random.default_rng(seed).choice(idx_in, min(max_in, len(idx_in)), replace=False)
        idx_out = np.flatnonzero(comp != c)
        idx_out = np.random.default_rng(seed).choice(idx_out, min(n_out, len(idx_out)), replace=False)
        jobs.append(delayed(_component_job)([db[i] for i in idx_in], [db[i] for i in idx_out],
                                            rel_sup, max_cand))
    return dict(zip(order, Parallel(n_jobs=n_jobs, verbose=5)(jobs)))

# the local mining is the point: a global 4% floor would bury the 6% components entirely
motifs = component_motifs()
for c, (_, name, role) in COMP.items():
    d = motifs[c]
    print(f"\n===== component {c} [{role}]: {name}  (n={(comp == c).sum()}, mined on {min(3000, (comp == c).sum())})")
    if len(d):
        display(d.head(4))
    else:
        print("  no ordered pattern of length >= 3 reaches the in-component support floor")


Each component now has **candidate motifs attached** — an ordered sequence, not just a set of
prevalent codes. Things to check:

- Does **c3** read as a workup path (troponin, EKG, imaging, angiography, in that order) and does
  **c0** read as a slower outpatient path (EKG, stress test, labs)?
- Do `anxiety` / `panic_attack` show up as a motif *before* troponin in **c3**, or only after it?
  Without visits, a "panic in the ED" story has to be carried by the workup codes alone.
- Do the background bundles (c2, c5, c6, c7) stay generic?

### A second lens: the Day 1 psych / cardiac probe

`chest_pain_labels()` is a direct rule-based split built from corroborating co-evidence. It is a
probe, not the structure: the labels come from `PSYCH_TAGS` / `CARDIAC_TAGS`, so a motif made of
those same tags partly rediscovers the label rule. Mined on a random 5,000-patient sample, since
every candidate is scanned against every sequence.


In [ ]:
weak = eo.chest_pain_labels(events)
wl   = weak.set_index('person_id').chest_pain_label

# All patients here have chest_pain by cohort construction, so no filtering needed. LOCAL mining on a
# fixed random sample: candidates x sequences scans are the cost on a cohort this size.
rng   = np.random.default_rng(0)
samp  = rng.choice(len(db), min(5_000, len(db)), replace=False)
db_s  = [db[i] for i in samp]
y_s   = np.array([wl.get(pids[i], 'unlabelled') for i in samp])
print("sampled patients:", len(db_s),
      "  labelled:", int((y_s != 'unlabelled').sum()),
      "  ", dict(zip(*np.unique(y_s, return_counts=True))))

pats_cp = prefixspan(db_s, max(3, int(0.08 * len(db_s))), max_len=4)
C_cp = [tuple(p) for p in pats_cp[pats_cp.pattern.map(len) >= 3]
        .sort_values('support', ascending=False).head(60).pattern]

for g in ['psych', 'cardiac']:
    print(f"\n=== {g} evidence")
    display(discriminative(y_s == g, C_cp, sequences_db=db_s, min_in=0.10).head(6))


**This is the deliverable.** Ordered patterns, with in-group and out-group support, that read
as sentences. No transformer, no embeddings, no non-commercial licence to negotiate.

## 3 · Interval relations — when order is not enough

Order says "A before B." It does not say "A the same week" versus "A two years before B,"
which is the actual dx_9487 discriminator: is the anxiety history old and unrelated, or did
a cardiac workup follow within days. Allen's interval algebra defines 13 relations; for
point events on dates, a **signed gap** carries most of the value.

In [ ]:
gaps = eo.signed_gap(events, 'chest_pain', 'troponin')
gaps = gaps.merge(weak, on='person_id', how='left')
gaps['chest_pain_label'] = gaps['chest_pain_label'].fillna('unlabelled')

plt.figure(figsize=(8.5, 3))
for name, g in gaps.groupby('chest_pain_label'):
    plt.scatter(g.days_a_to_b, np.random.normal(0, .05, len(g)),
                s=26, alpha=.75, label=f"{name} (n={len(g)})")
plt.axvline(0, c='k', lw=1); plt.yticks([])
plt.xlabel('days from chest_pain to troponin   (negative = troponin first)')
plt.legend(fontsize=8); plt.title('one signed integer, most of the workup-timing signal')
plt.tight_layout(); plt.show()

display(gaps.groupby('chest_pain_label').days_a_to_b
            .describe()[['count', 'mean', '50%']].round(0))

**One signed integer.** Worth remembering the next time the conversation turns to four weeks
of GPU pretraining.

Two cautions on real data. This feature is only meaningful where lookback covers the earlier
event — otherwise you are measuring enrolment, not biology. And for patients with no
troponin at all the feature is NULL, which is not the same as zero; carry an explicit
`no_workup_in_lookback` flag rather than imputing (this is exactly what
`cardiac_workup_outcome()` does formally in Day 5).


### 3b · Does psych evidence come before or after troponin?

Motif mining only surfaces patterns above a support floor, and psychiatric care is a minority inside
every bundle, so it can miss exactly this question. This asks it directly, with no floor: for each
patient with both, the days from their first psych evidence to their first troponin.

- **Psych evidence** = an anxiety or panic diagnosis, or a psychiatric evaluation / psychotherapy code
  (`psych_eval_therapy`). SSRIs and benzodiazepines are left out: they are prescribed for many reasons.
- Positive = psych evidence **first**, then troponin. Negative = troponin first.
- Patients with psych evidence and no troponin are not in this comparison (no workup in the data).
- Only meaningful where history covers the earlier event, as in section 3.


In [ ]:
PSYCH = ['anxiety', 'panic_attack', 'psych_eval_therapy']
ev    = events[['person_id', 'event_date', 'tag']]                  # no copy of the full frame
first = lambda tags: ev.loc[ev.tag.isin(tags)].groupby('person_id').event_date.min()

both = pd.concat([first(PSYCH).rename('psych'), first(['troponin']).rename('trop')], axis=1).dropna()
both['days_psych_to_trop'] = (both.trop - both.psych).dt.days
both['bundle'] = pd.Series(comp, index=pids).reindex(both.index)
both = both[both.bundle >= 0]                                   # drop patients with no assigned bundle

def timing(d):
    g = d.days_psych_to_trop
    return pd.Series({'n': len(d), 'psych first %': round((g > 0).mean() * 100, 1),
                      'same day %': round((g == 0).mean() * 100, 1),
                      'troponin first %': round((g < 0).mean() * 100, 1),
                      'median days': g.median()})

tab = both.groupby('bundle').apply(timing)
tab.index = [f"c{int(b)} {COMP[int(b)][1][:34]}" for b in tab.index]
tab.loc['ALL (psych + troponin)'] = timing(both)
display(tab.sort_values('n', ascending=False))

c3d = both[both.bundle == 3].days_psych_to_trop
plt.figure(figsize=(8.5, 3))
plt.hist(both.days_psych_to_trop.clip(-1500, 1500), bins=60, alpha=.5, density=True, label=f"all (n={len(both):,})")
plt.hist(c3d.clip(-1500, 1500), bins=60, alpha=.5, density=True, label=f"c3 acute / ED (n={len(c3d):,})")
plt.axvline(0, c='k', lw=1); plt.yticks([])
plt.xlabel('days from first psych evidence to first troponin   (negative = troponin first; clipped at +-1500)')
plt.legend(fontsize=8); plt.title('psych evidence vs troponin timing')
plt.tight_layout(); plt.show()


## 4 · Trajectory graph — the readable artifact

Aggregate consecutive transitions into a weighted directed graph. This is the closest thing
to a chest-pain triage flowchart, but derived from observed transitions rather than a
guideline document — Ethos has no chest-pain ruleset to start from, so the agreed method is
to start from the data and check clinical plausibility afterwards.

In [ ]:
import networkx as nx

FOCUS = {'chest_pain', 'ekg', 'troponin', 'stress_test',
         'coronary_angio', 'beta_blocker', 'statin', 'cad',
         'anxiety', 'panic_attack', 'ssri', 'benzodiazepine',
         'gerd', 'costochondritis'}

db_cp = [[c for c in seqs[p] if c in FOCUS] for p in pids if 'chest_pain' in seqs[p]]
print(f"{len(db_cp)} chest pain patients")

cnt = collections.Counter()
for s in db_cp:
    for u, v in zip(s, s[1:]):
        cnt[(u, v)] += 1
G = nx.DiGraph()
for (u, v), w in cnt.items():
    if w >= max(5, int(0.02 * len(db_cp))):
        G.add_edge(u, v, weight=w)
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

plt.figure(figsize=(10, 6.5))
pos = nx.spring_layout(G, seed=3, k=1.1)
w   = np.array([G[u][v]['weight'] for u, v in G.edges()], dtype=float)
nx.draw_networkx_nodes(G, pos, node_size=1600, node_color='#DCE6EF')
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, width=1 + 4 * w / w.max(), alpha=.5,
                       arrowsize=12, connectionstyle='arc3,rad=0.08')
plt.title('chest pain trajectory graph -- observed transitions')
plt.axis('off'); plt.tight_layout(); plt.show()

Two arms should be visible out of `chest_pain`: one through troponin, stress test,
angiography and statins; the other through anxiety, panic_attack, ssri and benzodiazepine.
Whichever arm dominates numerically is the answer to the working hypothesis — let the graph
say which, rather than assuming.

## Exercises

1. Lower `MIN_SUP` to 1%. How many patterns, and is closed-pattern pruning enough?
2. Extend `prefixspan` to enforce a maximum time gap between consecutive items. Which
   patterns die? (This is the single highest-value extension for claims.)
3. Add `gerd` and `costochondritis` to the trajectory graph's `FOCUS` set with a lower edge
   threshold. Do either show up as a real third arm, or are they noise?
4. Re-run the whole notebook with `tag_events(..., fallback='drop')` so only curated
   codesets survive. Cleaner patterns, but what did you lose?
5. Write the top motif per group (psych, cardiac) as a named plain-English rule. **That
   list is the motif catalog** — the thing to actually put in front of RUW.

## Carry forward

A motif catalog for dx_9487 triage, obtained without a foundation model.